# Two-stage LoRA, part two, LB 0.620

Our [first notebook](https://www.kaggle.com/code/nishantdahal/two-stage-lora-on-the-public-pool) scored 0.592 with two Qwen3 LoRAs trained on 35,000 public passages. This one scores 0.620 with the same recipe. Three things changed. The public pool got bigger and cleaner, two new members joined, and fixed a tokenizer bug that only shows up on Kaggle. Below is what each change did, on the same 5-fold CV split by prompt.

| Member | Trains on | First notebook | This notebook |
|---|---|---|---|
| n-gram SVC | train plus the public pool, fit here | 0.435 | 0.473 |
| Qwen3-14B LoRA, 27 of 40 layers | public pool, then train | 0.569 | 0.611 |
| Qwen2.5-14B LoRA, 32 of 48 layers | public pool, then train | | 0.569 |
| DeBERTa-v3-large, full fine tune | public pool, then train | | 0.546 |
| Blend with Sinkhorn, CV | | 0.593 | **0.642** |
| Public LB, with the centred twin | | 0.592 | **0.620** |

Member rows are balanced accuracy of that member alone, out of fold. The first notebook also had a Qwen3-8B LoRA at 0.569, which is gone now.

Everything the notebook needs is in the dataset [NTL two-stage LoRA part two](https://www.kaggle.com/datasets/nishantdahal/ntl-handwriting-blend). The base models are Kaggle's copies of Qwen3 14B Base and Qwen2.5 14B. Inference only. The commit run on the public test takes about 15 minutes on two T4s.

In [1]:
import gc, glob, json, os, re, shutil, time

# The scoring run has no internet, so transformers must never try to reach the Hub.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import numpy as np
import pandas as pd
import torch
from torch import nn

# Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

T0 = time.time()
INPUT_DIR = "/kaggle/input"

# Blend weights. Every member adds weight * log(probability) to the score.
W_SVC = 1.0
LORAS = {  # checkpoint folder, then blend weight, centred twin weight, batch size and the base model to look for
    "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
    "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
}
W_DEBERTA = 0.25
TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

# Column order of every checkpoint's head. Each checkpoint's meta.json is checked against it below.
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
K = len(LABELS)


def find(pattern):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(pattern)
    return hits[0]


def softmax(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def vote(name, P, w, rows=slice(None)):
    """Add one member's weighted log probabilities to the running score."""
    LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
    print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
test["text"] = test["text"].fillna("").astype(str)
y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
LOGIT = np.zeros((len(test), K))
print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

train (2699, 4) test (3, 2) gpus 2


## 1. n-gram SVC

The same four TF-IDF views as the first notebook. Raw characters, words with punctuation, word shapes, and small words with every content word replaced by CONTENT. The same LinearSVC with C of 0.5 and balanced class weights. The only change is the pool it fits on, 75,783 public passages instead of 34,940. That alone moved it from 0.435 to 0.473.

In [2]:
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


def shape_text(t):
    """Every run of letters becomes X or x, so only the rhythm is left."""
    return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


def small_words_text(t):
    """Keep the small words and the punctuation, and replace every other word by CONTENT."""
    return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                    for w in re.findall(r"\w+|[^\w\s]", t))


pool = pd.read_parquet(find("pool_v4.parquet"))
fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
A, B = [], []
for view, lo, hi, max_features, weight in VIEWS:
    prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
    analyzer = "char" if view == "shape" else "word" if view == "small" else view
    opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                max_features=max_features, dtype=np.float32)
    if analyzer == "word":
        opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
    vec = TfidfVectorizer(**opts)
    A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
    B.append(vec.transform([prep(t) for t in test.text]) * weight)
svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
vote("svc", softmax(svc.decision_function(hstack(B, format="csr"))), W_SVC)
del A, B, svc, pool
gc.collect()

svc: weight 1.0, done after 393s


0

## 2. The pool, the change that mattered most

Most of the gain came from data, not from models. On the old pool of 35,000 passages, four backbones finished at the same 0.569. Qwen3-8B, Qwen3-14B, Qwen2.5-7B and Qwen2.5-14B. The same Qwen3-14B with the same recipe on the new pool finished at 0.611.

You can see it before the host data even comes in. After stage one alone, trained only on public passages, the 14B's head scores 0.572 on the host passages. With the old pool it scored 0.493. The backbone did not change. The pool did.

The pool the 14B read in stage one has 83,827 passages from 26 public datasets. It keeps the filters of the first notebook. One paragraph, 30 to 300 words, no markdown, no duplicates, nothing identical to a train passage, and no punctuation mark that a model never uses in the host data. Two filters are new.

- **One source is out.** A set of claude-fable-5 passages written with a system prompt. They read nothing like the host's Fable passages.
- **Public passages too close to a train passage are out.** 1,153 passages whose content TF-IDF cosine with some train passage is 0.3 or more. Some public datasets answer prompts rebuilt from the training data. If stage one has already seen a host prompt, the out of fold score of stage two is too kind. This filter keeps the CV honest. It does not raise it.

The dataset ships the 75,783 passage version that the SVC fits on. It has the first filter but not the second.

## 3. Two new members

**Qwen2.5-14B LoRA.** The same recipe as the Qwen3 LoRAs, on the first 32 of 48 layers, pooled at layers 3, 5, 8, 11, 16, 21, 27 and 32. Stage one ran on the old 35,000 passage pool. Alone it ties the Qwen3 models at 0.569. Its top prediction agrees with theirs on about 77 percent of passages, which is about as often as they agree with each other. So it is not a new view, and it gets a small weight, 0.2.

**DeBERTa-v3-large, fully fine tuned.** No LoRA here, all the weights train. The same two stages. Stage one is 2 epochs over 61,486 public passages with class weights, at learning rate 0.000015 for the encoder and 0.001 for the head. Stage two is 3 epochs on train at 0.00001 and 0.0005. Max length 384, the mean of the last layer, one linear head. The bigger pool moved it from 0.512 to 0.546 alone.

## 4. LoRA inference, and a bug that only shows up on Kaggle

The inference code is the first notebook's. Keep the first layers, merge the adapter into the weights, average the hidden states at 8 depths without the first token, one LayerNorm per depth, one linear head. Two things are new.

**The end token.** Every passage gets an end token appended, in training and at inference. Our training tokenizer used `<|endoftext|>`, id 151643. Kaggle's copy of Qwen3-14B-Base ships the tokenizer config of the chat model, whose end token is `<|im_end|>`, id 151645. So on Kaggle every passage ended with a token the model never saw there in training, and the 14B's top prediction changed on about 14 percent of passages compared with the same checkpoint run locally. Nothing crashes and nothing warns. The fix is one line. Use the id saved in the checkpoint's meta.json. Kaggle's Qwen2.5-14B does not have this problem. If you run Qwen3-14B-Base from Kaggle Models, print which end token you get.

**T4 numerics and memory.**

- We tell cuBLAS to keep fp32 sums in fp16 matmuls. It costs nothing, so we set it.
- A batch that overflows fp16 is redone in bf16, as before.
- A batch that runs out of memory on the 15 GB T4 is redone one passage at a time.
- There is no try except block around the members. In an earlier submission a handler like that skipped a member that ran out of memory on the T4. The run still finished and scored, a few points lower, and we only found out from the log. A member that breaks should stop the notebook.

In [3]:
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer


def find_base(spec):
    """The attached base model whose config.json matches spec."""
    for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    raise FileNotFoundError(f"attach the base model {spec}")


def load_trunk(base_dir, keep, dtype):
    """The first keep decoder layers, split over both T4s when they do not fit on one."""
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
    hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
    q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
    gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
    if gib > 13 and torch.cuda.device_count() > 1:
        mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
    else:
        model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
    model.norm = nn.Identity()   # the checkpoints were trained without the final norm
    print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    """W becomes W + B A alpha / r for every adapted linear layer."""
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        lin = modules[name[name.index("layers."):]]
        Bm = sd[k.replace(".lora_A.", ".lora_B.")]
        delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
        lin.weight.add_(delta.to(lin.weight.dtype))


@torch.no_grad()
def lora_features(ckpt_dir, base_dir, texts, bs):
    """Pooled features Z, one row per passage, and the checkpoint's heads."""
    meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
    cfg = meta["cfg"]
    dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = load_trunk(base_dir, cfg["keep"], dtype)
    merge_lora(model, os.path.join(ckpt_dir, "adapter"))
    heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
    layers, H = cfg["layers"], model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
    # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the chat model's end token.
    eos_id = cfg.get("eos_id", tok.eos_token_id)
    print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
    Z = np.zeros((len(ids), H * len(layers)), np.float32)

    def embed(idx):
        enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
        hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
        if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            model.to(dtype)
        m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
        m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
        den = m.sum(1).clamp(min=1)
        Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

    for s in range(0, len(order), bs):
        idx = order[s:s + bs]
        try:
            embed(idx)
        except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4, so go one passage at a time
            torch.cuda.empty_cache()
            for i in idx:
                embed([i])
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

## 5. Centred twin

As in the first notebook. Cluster the test passages by content TF-IDF. In every cluster of 2 to 18 passages, subtract the cluster mean from each LoRA's features and read the rest with the centred head, at weight 0.1 per LoRA. It is not in the CV numbers above, because CV has no test clusters.

In [4]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform

if len(test) >= 2:
    Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
    D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
    np.fill_diagonal(D, 0)
    groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
    del D
else:
    groups = np.ones(len(test), dtype=int)
size = np.bincount(groups)[groups]
twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

for ckpt_name, mc in LORAS.items():
    ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
    Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
    vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
    if twin_rows.any():
        Zc = Z.copy()
        for g in np.unique(groups[twin_rows]):
            Zc[groups == g] -= Zc[groups == g].mean(0)
        centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
        vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
    del Z
    gc.collect()

topic clusters: 3 passages alone, 0 in clusters of 2 to 18


Loading weights:   0%|          | 0/299 [00:00<?, ?it/s]

[transformers] Qwen3Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen-3/transformers/14b-base/1
Key                                                    | Status     |  | 
-------------------------------------------------------+------------+--+-
model.layers.{27...39}.post_attention_layernorm.weight | UNEXPECTED |  | 
model.layers.{27...39}.input_layernorm.weight          | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.o_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.mlp.gate_proj.weight            | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.v_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.mlp.down_proj.weight            | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.q_proj.weight         | UNEXPECTED |  | 
model.layers.{27...39}.mlp.up_proj.weight              | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.q_norm.weight         | UNEXPECTED |  | 
model.layers.{27...39}.self_attn.k_proj.weight         | UNEXPECTED |  | 
model.la

trunk of 27 layers, 18.1 GiB of weights
end token 151643, this tokenizer's is 151645
lora_qwen3_14b: weight 0.4, done after 483s


Loading weights:   0%|          | 0/386 [00:00<?, ?it/s]

[transformers] Qwen2Model LOAD REPORT from: /kaggle/input/models/qwen-lm/qwen2.5/transformers/14b/1
Key                                                    | Status     |  | 
-------------------------------------------------------+------------+--+-
model.layers.{32...47}.post_attention_layernorm.weight | UNEXPECTED |  | 
model.layers.{32...47}.input_layernorm.weight          | UNEXPECTED |  | 
model.layers.{32...47}.self_attn.q_proj.weight         | UNEXPECTED |  | 
model.layers.{32...47}.mlp.up_proj.weight              | UNEXPECTED |  | 
model.layers.{32...47}.self_attn.v_proj.bias           | UNEXPECTED |  | 
model.layers.{32...47}.self_attn.k_proj.weight         | UNEXPECTED |  | 
model.layers.{32...47}.self_attn.o_proj.weight         | UNEXPECTED |  | 
model.layers.{32...47}.mlp.gate_proj.weight            | UNEXPECTED |  | 
model.layers.{32...47}.self_attn.k_proj.bias           | UNEXPECTED |  | 
model.layers.{32...47}.mlp.down_proj.weight            | UNEXPECTED |  | 
model.layers

trunk of 32 layers, 17.9 GiB of weights
end token 151643, this tokenizer's is 151643
lora_qwen25_14b: weight 0.2, done after 613s


## 6. DeBERTa

The checkpoint is copied to local disk before loading. Loading it straight from the dataset mount took 16 minutes in our runs. The copy plus the load take 25 seconds.

In [5]:
ck = os.path.dirname(find("deberta_v3_large/meta.json"))
if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
    shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
    ck = "/tmp/deberta_v3_large"
meta = json.load(open(os.path.join(ck, "meta.json")))
assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
head = load_file(os.path.join(ck, "head.safetensors"))
W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
order = np.argsort([len(x) for x in ids], kind="stable")
Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
with torch.inference_mode():
    for s in range(0, len(order), 32):
        idx = order[s:s + 32]
        bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        m = bt["attention_mask"].unsqueeze(-1).float()
        Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
del enc
gc.collect()
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

deberta: weight 0.25, done after 646s


## 7. Blend and Sinkhorn

A weighted sum of log probabilities. SVC 1.0, Qwen3-14B 0.4, Qwen2.5-14B 0.2, DeBERTa 0.25. These are round numbers chosen on out of fold predictions. Fitting them finer on 2,699 passages mostly fits noise. Then Sinkhorn balancing to equal class totals as in the first notebook, 300 iterations, skipped on the 3 row public test.

In [6]:
def sinkhorn(P, iters=300):
    """One scale per class, so that every class gets an equal share of the probability. Rows are renormalised."""
    c = np.ones(K)
    target = np.full(K, len(P) / K)
    for _ in range(iters):
        R = P * c
        R /= R.sum(1, keepdims=True)
        c *= target / R.sum(0)
    R = P * c
    return R / R.sum(1, keepdims=True)


P = softmax(LOGIT)
if len(test) >= 50:   # skip on the 3 row public test.csv
    P = sinkhorn(P)
sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
sub.to_parquet("submission.parquet", index=False)
print(sub["model"].value_counts().to_string())
print(f"done in {time.time() - T0:.0f}s")

model
claude-opus-4-8           1
gpt-5.6-sol               1
gemini-3.1-pro-preview    1
done in 647s


## CV says 0.642, LB says 0.620

The first notebook lost almost nothing between CV and LB, 0.593 against 0.592. This one loses 2.2 points. What changed between the two is mostly the pool. Our CV only tests on host passages, but the members learned much of what they know from public passages written with other settings. Some of that seems not to carry over to the hidden test. We have guesses and no clean answer yet. Part of the gap is plain noise too, since the public LB is only a slice of the test.

## What did not help

- **More parameters.** See section 2.
- **Averaging seeds**, or averaging the LoRA weights of several runs. 0.1 points at most.
- **A Gemma-4-12B backbone** with the same recipe.
- **Gates for one family**, for example a separate judge just for the Claude and GLM passages.
- **At most one passage per model inside a topic cluster.** Better in simulation, worse on the LB.
- **More stage two epochs**, or public passages mixed into stage two.
- **Stage two on short windows** cut from the host passages. Short passages are hard for every member, and the windows did not change that.
- **A stacker** instead of fixed weights.

None of it helped the two hardest pairs, `glm-5.3` against `glm-5.3-flash` and `gemini-3.7-flash` against `gemini-3.8-flash`. Every member gets stuck at about the same place on them.

## Credits

Public passages, CC BY 4.0 unless another license is given.

- [seyitkaangunes/ntl-extra-api-passages](https://www.kaggle.com/datasets/seyitkaangunes/ntl-extra-api-passages)
- [willrice/name-that-llm-openrouter-harvest](https://www.kaggle.com/datasets/willrice/name-that-llm-openrouter-harvest)
- [lxdlxdlxd/ntl-synth-generations-v1](https://www.kaggle.com/datasets/lxdlxdlxd/ntl-synth-generations-v1)
- [prvsiyan/prvsiyans-extended-name-that-llm-dataset](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset)
- [alperen5252525/ntl-vertex-gemini-proxy-gpt-generations](https://www.kaggle.com/datasets/alperen5252525/ntl-vertex-gemini-proxy-gpt-generations)
- [haileycheng/name-that-llm-same-prompt-passages](https://www.kaggle.com/datasets/haileycheng/name-that-llm-same-prompt-passages)
- [kaplinskiy/ntl-gpt-proxy-passages](https://www.kaggle.com/datasets/kaplinskiy/ntl-gpt-proxy-passages)
- [lucasboesen/ntl-free-proxy-generations](https://www.kaggle.com/datasets/lucasboesen/ntl-free-proxy-generations)
- [abdullah1afify/ntl-generations](https://www.kaggle.com/datasets/abdullah1afify/ntl-generations)
- [dophuchung/ntl-proxy-generations](https://www.kaggle.com/datasets/dophuchung/ntl-proxy-generations)
- [ziyuzhou1207/ntl-zz-proxy-generations](https://www.kaggle.com/datasets/ziyuzhou1207/ntl-zz-proxy-generations)
- [keshavatearth/ntl-gpt-new-prompts-v1](https://www.kaggle.com/datasets/keshavatearth/ntl-gpt-new-prompts-v1)
- [michaelsemenoff/ntl-own-extra-v1](https://www.kaggle.com/datasets/michaelsemenoff/ntl-own-extra-v1)
- [jadkutbi/ntl-own-proxy-generations](https://www.kaggle.com/datasets/jadkutbi/ntl-own-proxy-generations)
- [pardheev/ntl-proxy-extra-passages](https://www.kaggle.com/datasets/pardheev/ntl-proxy-extra-passages)
- [abhinavsaipodugu/ntl-proxy-gpt-gemini-passages](https://www.kaggle.com/datasets/abhinavsaipodugu/ntl-proxy-gpt-gemini-passages)
- [zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations](https://www.kaggle.com/datasets/zakirkhanaleemi/name-that-llm-benchmarks-proxy-generations)
- [zbigniewkirchner/ntl-extra-gpt-passages](https://www.kaggle.com/datasets/zbigniewkirchner/ntl-extra-gpt-passages)
- [arxitekton/ntl-claude-gen-v1](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v1)
- [arxitekton/ntl-claude-gen-v2](https://www.kaggle.com/datasets/arxitekton/ntl-claude-gen-v2)
- [maniceet/name-that-llm-openrouter-openai-generations](https://www.kaggle.com/datasets/maniceet/name-that-llm-openrouter-openai-generations)
- [dedruck/ntl-proxy-gen](https://www.kaggle.com/datasets/dedruck/ntl-proxy-gen)
- [prvsiyan/ntl-own-prompt-generations-v1](https://www.kaggle.com/datasets/prvsiyan/ntl-own-prompt-generations-v1), CC BY-SA 4.0
- [drjcfmd/ntl-generated-v2](https://www.kaggle.com/datasets/drjcfmd/ntl-generated-v2), CC BY-SA 4.0
- [illiadolenko/ntl-generated-corpus-v1](https://www.kaggle.com/datasets/illiadolenko/ntl-generated-corpus-v1), CC0
- [aswathvanan/ntl-proxy-passages](https://www.kaggle.com/datasets/aswathvanan/ntl-proxy-passages), CC BY-SA 4.0, only in stage one of the Qwen3-14B LoRA

Ideas and recipes.

- The LoRA24 recipe from [lightningv08/ntl-lora24-fivefold-training](https://www.kaggle.com/code/lightningv08/ntl-lora24-fivefold-training) and [lucasboesen/name-that-llm-train-lora24-s7](https://www.kaggle.com/code/lucasboesen/name-that-llm-train-lora24-s7)
- The style aware n-gram views from [yangkuangou](https://www.kaggle.com/yangkuangou)'s notebook
- Adaptive centering, the centred twin and Sinkhorn balancing from [lucasboesen/lora-x-adaptive-centering](https://www.kaggle.com/code/lucasboesen/lora-x-adaptive-centering)

Models. Qwen3-14B-Base and Qwen2.5-14B by the Qwen team, Apache 2.0. DeBERTa-v3-large by Microsoft, MIT.